In [6]:
from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = (
    Path("..")
    if Path.cwd().name == "notebooks"
    else Path(".")
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "portfolio_returns.csv"
)

EXPECTED_COLUMNS = [
    "date",
    "HPG_simple_return",
    "FPT_simple_return",
    "MWG_simple_return",
    "portfolio_simple_return",
    "portfolio_log_return",
]


if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Canonical portfolio dataset not found: {DATA_PATH.resolve()}"
    )


portfolio_data = pd.read_csv(DATA_PATH)

if portfolio_data.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "Unexpected portfolio dataset schema.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Received: {portfolio_data.columns.tolist()}"
    )


portfolio_data["date"] = pd.to_datetime(
    portfolio_data["date"],
    errors="raise",
)

portfolio_data = (
    portfolio_data
    .sort_values("date")
    .reset_index(drop=True)
)


duplicate_dates = int(
    portfolio_data["date"].duplicated().sum()
)

missing_values = (
    portfolio_data[EXPECTED_COLUMNS]
    .isna()
    .sum()
)

portfolio_returns = pd.to_numeric(
    portfolio_data["portfolio_simple_return"],
    errors="raise",
).astype("float64")

finite_returns = np.isfinite(
    portfolio_returns.to_numpy()
)

nonfinite_count = int(
    (~finite_returns).sum()
)


if portfolio_data.empty:
    raise ValueError(
        "Canonical portfolio dataset is empty."
    )

if duplicate_dates != 0:
    raise ValueError(
        f"Duplicate portfolio dates detected: {duplicate_dates}"
    )

if missing_values.sum() != 0:
    raise ValueError(
        "Missing values detected in canonical portfolio dataset:\n"
        f"{missing_values[missing_values > 0]}"
    )

if nonfinite_count != 0:
    raise ValueError(
        "Portfolio simple returns contain "
        f"{nonfinite_count} non-finite values."
    )


print("--- HISTORICAL VAR INPUT AUDIT ---")
print(f"Data path: {DATA_PATH.resolve()}")
print(f"Rows: {len(portfolio_data)}")
print(
    "Date range:",
    portfolio_data["date"].min().date(),
    "->",
    portfolio_data["date"].max().date(),
)
print(f"Duplicate dates: {duplicate_dates}")
print(f"Missing values: {int(missing_values.sum())}")
print(f"Non-finite portfolio returns: {nonfinite_count}")
print(
    "Portfolio return range:",
    f"{portfolio_returns.min():.6%}",
    "->",
    f"{portfolio_returns.max():.6%}",
)

print("\nCanonical VaR series:")
print(portfolio_returns.head())


--- HISTORICAL VAR INPUT AUDIT ---
Data path: C:\Users\Admin\Downloads\portfolio-var-risk-system\data\processed\portfolio_returns.csv
Rows: 1637
Date range: 2020-01-03 -> 2026-07-28
Duplicate dates: 0
Missing values: 0
Non-finite portfolio returns: 0
Portfolio return range: -6.983986% -> 6.887777%

Canonical VaR series:
0   -0.008343
1   -0.007326
2    0.004948
3   -0.019132
4    0.017507
Name: portfolio_simple_return, dtype: float64


In [7]:
synthetic_returns = pd.Series(
    [
        0.012,
        -0.020,
        0.008,
        -0.080,
        0.015,
        -0.010,
        0.025,
        -0.030,
        0.005,
        -0.050,
        0.018,
        -0.015,
        0.030,
        -0.025,
        0.010,
        -0.005,
        0.022,
        -0.040,
        0.003,
        0.014,
        -0.035,
    ],
    name="portfolio_simple_return",
    dtype="float64",
)

confidence_level = 0.95
alpha = 1.0 - confidence_level

sorted_returns = (
    synthetic_returns
    .sort_values()
    .reset_index(drop=True)
)

n_observations = len(sorted_returns)

quantile_position = (
    (n_observations - 1)
    * alpha
)

lower_index = int(
    np.floor(quantile_position)
)

upper_index = int(
    np.ceil(quantile_position)
)

interpolation_weight = (
    quantile_position
    - lower_index
)

lower_return = float(
    sorted_returns.iloc[lower_index]
)

upper_return = float(
    sorted_returns.iloc[upper_index]
)

manual_quantile = (
    lower_return
    + interpolation_weight
    * (upper_return - lower_return)
)

numpy_quantile = float(
    np.quantile(
        synthetic_returns.to_numpy(),
        alpha,
        method="linear",
    )
)

historical_var = max(
    0.0,
    -manual_quantile,
)

absolute_difference = abs(
    manual_quantile
    - numpy_quantile
)


assert n_observations == 21

assert np.isclose(
    manual_quantile,
    -0.05,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    historical_var,
    0.05,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    manual_quantile,
    numpy_quantile,
    rtol=0.0,
    atol=1e-12,
)


print("SYNTHETIC HISTORICAL VAR CHECK")

print("\nSorted returns:")
print(
    sorted_returns.to_string(
        index=True,
    )
)

print("\nQuantile diagnostics:")
print(f"Observations: {n_observations}")
print(
    f"Confidence level: "
    f"{confidence_level:.2%}"
)
print(f"Alpha: {alpha:.2%}")
print(
    f"Quantile position h: "
    f"{quantile_position:.6f}"
)
print(f"Lower index: {lower_index}")
print(f"Upper index: {upper_index}")
print(
    f"Interpolation weight: "
    f"{interpolation_weight:.6f}"
)

print("\nHistorical Simulation result:")
print(
    f"Manual q05: "
    f"{manual_quantile:.6%}"
)
print(
    f"NumPy q05: "
    f"{numpy_quantile:.6%}"
)
print(
    f"Historical VaR 95%: "
    f"{historical_var:.6%}"
)
print(
    f"Absolute difference: "
    f"{absolute_difference:.3e}"
)


SYNTHETIC HISTORICAL VAR CHECK

Sorted returns:
0    -0.080
1    -0.050
2    -0.040
3    -0.035
4    -0.030
5    -0.025
6    -0.020
7    -0.015
8    -0.010
9    -0.005
10    0.003
11    0.005
12    0.008
13    0.010
14    0.012
15    0.014
16    0.015
17    0.018
18    0.022
19    0.025
20    0.030

Quantile diagnostics:
Observations: 21
Confidence level: 95.00%
Alpha: 5.00%
Quantile position h: 1.000000
Lower index: 1
Upper index: 2
Interpolation weight: 0.000000

Historical Simulation result:
Manual q05: -5.000000%
NumPy q05: -5.000000%
Historical VaR 95%: 5.000000%
Absolute difference: 0.000e+00


In [8]:
def calculate_historical_var(
    returns: pd.Series,
    confidence_level: float = 0.95,
) -> dict[str, float | int]:
    """
    Calculate Historical Simulation VaR from a return sample.
    """
    if not isinstance(returns, pd.Series):
        raise ValueError(
            "Returns must be provided as a pandas Series."
        )

    if returns.empty:
        raise ValueError(
            "Returns cannot be empty."
        )

    if not isinstance(
        confidence_level,
        (int, float),
    ):
        raise ValueError(
            "Confidence level must be numeric."
        )

    confidence_level = float(
        confidence_level
    )

    if (
        not np.isfinite(confidence_level)
        or confidence_level <= 0.0
        or confidence_level >= 1.0
    ):
        raise ValueError(
            "Confidence level must be finite "
            "and strictly between 0 and 1."
        )

    try:
        numeric_returns = pd.to_numeric(
            returns,
            errors="raise",
        ).astype("float64")
    except (TypeError, ValueError) as error:
        raise ValueError(
            "Returns must contain only numeric values."
        ) from error

    if numeric_returns.isna().any():
        raise ValueError(
            "Returns cannot contain missing values."
        )

    if not np.isfinite(
        numeric_returns.to_numpy()
    ).all():
        raise ValueError(
            "Returns must contain only finite values."
        )

    alpha = (
        1.0
        - confidence_level
    )

    quantile_return = float(
        np.quantile(
            numeric_returns.to_numpy(),
            alpha,
            method="linear",
        )
    )

    historical_var = max(
        0.0,
        -quantile_return,
    )

    result = {
        "confidence_level": confidence_level,
        "alpha": alpha,
        "observations": len(numeric_returns),
        "quantile_return": quantile_return,
        "historical_var": historical_var,
    }

    return result


synthetic_var_result = calculate_historical_var(
    synthetic_returns,
    confidence_level=0.95,
)


assert synthetic_var_result[
    "observations"
] == 21

assert np.isclose(
    synthetic_var_result[
        "quantile_return"
    ],
    -0.05,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    synthetic_var_result[
        "historical_var"
    ],
    0.05,
    rtol=0.0,
    atol=1e-12,
)


print(
    "HISTORICAL VAR FUNCTION "
    "REGRESSION"
)

print(
    "Confidence level:",
    f"{synthetic_var_result['confidence_level']:.2%}",
)

print(
    "Alpha:",
    f"{synthetic_var_result['alpha']:.2%}",
)

print(
    "Observations:",
    synthetic_var_result[
        "observations"
    ],
)

print(
    "Quantile return:",
    f"{synthetic_var_result['quantile_return']:.6%}",
)

print(
    "Historical VaR:",
    f"{synthetic_var_result['historical_var']:.6%}",
)

HISTORICAL VAR FUNCTION REGRESSION
Confidence level: 95.00%
Alpha: 5.00%
Observations: 21
Quantile return: -5.000000%
Historical VaR: 5.000000%


In [9]:
def expect_value_error(
    case_name: str,
    function,
) -> bool:
    """
    Confirm that a validation case raises ValueError.
    """
    try:
        function()
    except ValueError as error:
        print(
            f"[PASS] {case_name}: "
            f"{error}"
        )
        return True

    print(
        f"[FAIL] {case_name}: "
        "ValueError was not raised."
    )
    return False


validation_results = []


# Valid case 1: known synthetic ground truth
valid_result = calculate_historical_var(
    synthetic_returns,
    confidence_level=0.95,
)

valid_ground_truth = (
    np.isclose(
        valid_result["quantile_return"],
        -0.05,
        rtol=0.0,
        atol=1e-12,
    )
    and np.isclose(
        valid_result["historical_var"],
        0.05,
        rtol=0.0,
        atol=1e-12,
    )
)

validation_results.append(
    valid_ground_truth
)

print(
    "[PASS] Synthetic ground truth"
    if valid_ground_truth
    else "[FAIL] Synthetic ground truth"
)


# Valid case 2: alternative confidence level

result_99 = calculate_historical_var(
    synthetic_returns,
    confidence_level=0.99,
)

valid_99 = (
    np.isclose(
        result_99["confidence_level"],
        0.99,
        rtol=0.0,
        atol=1e-12,
    )
    and np.isclose(
        result_99["alpha"],
        0.01,
        rtol=0.0,
        atol=1e-12,
    )
)

validation_results.append(
    valid_99
)

print(
    "[PASS] Alternative confidence level"
    if valid_99
    else "[FAIL] Alternative confidence level"
)


# Valid case 3: positive-return sample

positive_returns = pd.Series(
    [
        0.001,
        0.003,
        0.005,
        0.007,
        0.010,
    ],
    dtype="float64",
)

positive_result = calculate_historical_var(
    positive_returns
)

positive_var_valid = np.isclose(
    positive_result["historical_var"],
    0.0,
    rtol=0.0,
    atol=1e-12,
)

validation_results.append(
    positive_var_valid
)

print(
    "[PASS] Positive sample produces zero VaR"
    if positive_var_valid
    else "[FAIL] Positive sample produces zero VaR"
)


# Valid case 4: input immutability

returns_before = synthetic_returns.copy(
    deep=True
)

calculate_historical_var(
    synthetic_returns
)

input_unchanged = synthetic_returns.equals(
    returns_before
)

validation_results.append(
    input_unchanged
)

print(
    "[PASS] Input immutability"
    if input_unchanged
    else "[FAIL] Input immutability"
)


# Invalid cases
invalid_cases = [
    (
        "Non-Series input",
        lambda: calculate_historical_var(
            [0.01, -0.02, 0.03]
        ),
    ),
    (
        "Empty Series",
        lambda: calculate_historical_var(
            pd.Series(
                dtype="float64"
            )
        ),
    ),
    (
        "Non-numeric return",
        lambda: calculate_historical_var(
            pd.Series(
                [
                    0.01,
                    "invalid",
                    -0.02,
                ],
                dtype="object",
            )
        ),
    ),
    (
        "Missing return",
        lambda: calculate_historical_var(
            pd.Series(
                [
                    0.01,
                    np.nan,
                    -0.02,
                ],
                dtype="float64",
            )
        ),
    ),
    (
        "Positive infinity",
        lambda: calculate_historical_var(
            pd.Series(
                [
                    0.01,
                    np.inf,
                    -0.02,
                ],
                dtype="float64",
            )
        ),
    ),
    (
        "Negative infinity",
        lambda: calculate_historical_var(
            pd.Series(
                [
                    0.01,
                    -np.inf,
                    -0.02,
                ],
                dtype="float64",
            )
        ),
    ),
    (
        "Confidence level zero",
        lambda: calculate_historical_var(
            synthetic_returns,
            confidence_level=0.0,
        ),
    ),
    (
        "Confidence level one",
        lambda: calculate_historical_var(
            synthetic_returns,
            confidence_level=1.0,
        ),
    ),
    (
        "Confidence level NaN",
        lambda: calculate_historical_var(
            synthetic_returns,
            confidence_level=np.nan,
        ),
    ),
    (
        "Non-numeric confidence level",
        lambda: calculate_historical_var(
            synthetic_returns,
            confidence_level="0.95",
        ),
    ),
]


for case_name, case_function in invalid_cases:
    validation_results.append(
        expect_value_error(
            case_name,
            case_function,
        )
    )


passed_cases = int(
    sum(validation_results)
)

total_cases = len(
    validation_results
)


assert passed_cases == total_cases


print(
    "\nHISTORICAL VAR "
    "BEHAVIORAL VALIDATION"
)

print(
    f"Passed cases: "
    f"{passed_cases}/{total_cases}"
)

[PASS] Synthetic ground truth
[PASS] Alternative confidence level
[PASS] Positive sample produces zero VaR
[PASS] Input immutability
[PASS] Non-Series input: Returns must be provided as a pandas Series.
[PASS] Empty Series: Returns cannot be empty.
[PASS] Non-numeric return: Returns must contain only numeric values.
[PASS] Missing return: Returns cannot contain missing values.
[PASS] Positive infinity: Returns must contain only finite values.
[PASS] Negative infinity: Returns must contain only finite values.
[PASS] Confidence level zero: Confidence level must be finite and strictly between 0 and 1.
[PASS] Confidence level one: Confidence level must be finite and strictly between 0 and 1.
[PASS] Confidence level NaN: Confidence level must be finite and strictly between 0 and 1.
[PASS] Non-numeric confidence level: Confidence level must be numeric.

HISTORICAL VAR BEHAVIORAL VALIDATION
Passed cases: 14/14


In [11]:
STATIC_CONFIDENCE_LEVEL = 0.95

static_var_result = calculate_historical_var(
    portfolio_returns,
    confidence_level=STATIC_CONFIDENCE_LEVEL,
)

direct_alpha = (
    1.0
    - STATIC_CONFIDENCE_LEVEL
)

direct_quantile = float(
    np.quantile(
        portfolio_returns.to_numpy(),
        direct_alpha,
        method="linear",
    )
)

direct_var = max(
    0.0,
    -direct_quantile,
)

quantile_difference = abs(
    static_var_result["quantile_return"]
    - direct_quantile
)

var_difference = abs(
    static_var_result["historical_var"]
    - direct_var
)


assert static_var_result[
    "observations"
] == len(portfolio_returns)

assert len(
    portfolio_returns
) == 1637

assert np.isclose(
    static_var_result["quantile_return"],
    direct_quantile,
    rtol=0.0,
    atol=1e-12,
)

assert np.isclose(
    static_var_result["historical_var"],
    direct_var,
    rtol=0.0,
    atol=1e-12,
)


print(
    "STATIC HISTORICAL SIMULATION VAR"
)

print(
    "Observations:",
    static_var_result["observations"],
)

print(
    "Confidence level:",
    f"{static_var_result['confidence_level']:.2%}",
)

print(
    "Alpha:",
    f"{static_var_result['alpha']:.2%}",
)

print(
    "Sample start:",
    portfolio_data["date"].min().date(),
)

print(
    "Sample end:",
    portfolio_data["date"].max().date(),
)

print(
    "Minimum return:",
    f"{portfolio_returns.min():.6%}",
)

print(
    "Maximum return:",
    f"{portfolio_returns.max():.6%}",
)

print(
    "\nFunction result:"
)

print(
    "Empirical q05:",
    f"{static_var_result['quantile_return']:.6%}",
)

print(
    "Historical VaR 95%:",
    f"{static_var_result['historical_var']:.6%}",
)

print(
    "\nIndependent NumPy cross-check:"
)

print(
    "Direct q05:",
    f"{direct_quantile:.6%}",
)

print(
    "Direct VaR:",
    f"{direct_var:.6%}",
)

print(
    "Quantile absolute difference:",
    f"{quantile_difference:.3e}",
)

print(
    "VaR absolute difference:",
    f"{var_difference:.3e}",
)

STATIC HISTORICAL SIMULATION VAR
Observations: 1637
Confidence level: 95.00%
Alpha: 5.00%
Sample start: 2020-01-03
Sample end: 2026-07-28
Minimum return: -6.983986%
Maximum return: 6.887777%

Function result:
Empirical q05: -2.614281%
Historical VaR 95%: 2.614281%

Independent NumPy cross-check:
Direct q05: -2.614281%
Direct VaR: 2.614281%
Quantile absolute difference: 0.000e+00
VaR absolute difference: 0.000e+00
